# Feature - Education | Text Model

## Import Libraries

In [43]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split
from transformers import BertTokenizer, BertForSequenceClassification, TrainingArguments, Trainer
import torch
from torch.utils.data import Dataset

## Define folder paths

In [3]:
ROOT_DIR = "../../dataset/Dataset"
TEXT_TRAIN_CSV = os.path.join(ROOT_DIR, "Texts", "Education", "Education_Train.csv")
TEXT_TEST_CSV = os.path.join(ROOT_DIR, "Texts", "Education", "Education_Test.csv")

IMAGE_TRAIN = os.path.join(ROOT_DIR, "Images", "Education", "train")
IMAGE_TEST = os.path.join(ROOT_DIR, "Images", "Education", "test")

## Load Data

In [4]:
columns = [
    "image_name",
    "text_tr",
    "text_en",
    "education",
    "extra"
]

train_df = pd.read_csv(TEXT_TRAIN_CSV, header=None, names=columns)
test_df  = pd.read_csv(TEXT_TEST_CSV, header=None, names=columns)

In [5]:
train_df.head()

,image_name,text_tr,text_en,education,extra
0,205-5G-1242-F-H,Birinci olmak çok gurur verici ve mutlu bir şe...,Being the first is a very proud and happy thin...,Secondary,NaN
1,210-5C-818-F-H,Mutluluk hayattaki ışıktır,Happiness is the light in life,Secondary,NaN
2,214-6B-421-F-S,Mutsuzluk benim için üzüldükten sonra asla pes...,Unhappiness is never giving up after feeling s...,Secondary,NaN
3,206-6D-790-F-S,"Benim için üzüntü haksızlığa uğramak, dışlanma...","For me, sadness is being wronged, being exclud...",Secondary,NaN
4,203-5H-720-F-H,Benim için mutluluk dünyada kötülüğün olmadığı...,"For me, happiness is a world where there is no...",Secondary,NaN


In [6]:
test_df.head()

,image_name,text_tr,text_en,education,extra
0,205-5D-417-F-H,"Ağaçlar, kuşlar ve en büyük mutluluğumdur","Trees, birds and my greatest happiness",Secondary,NaN
1,207-6H-282-M-H,Doğa deniz ve yeşil yerler beni mutluluğumdur,"Nature, sea and green places are my happiness",Secondary,NaN
2,280-5M-84-F-H,Bu resimde bana arkadaşına hediye alan ve bu k...,"In this picture, It tells me that she bought a...",Secondary,NaN
3,250-5C-1276-M-S,Pazar dönüşündeki yağmur,Rain on Sunday return,Secondary,NaN
4,210-5C-786-F-S,"Keşke o kötülüğü yapmasaydım, yalnız olmazdım","I wish I had not done that evil, I would not b...",Secondary,NaN


## Data Preprocessing

### Handle Invalid Values

In [7]:
train_df['extra'].value_counts()

extra
something joyful love    1
Name: count, dtype: int64

In [8]:
test_df['extra'].value_counts()

Series([], Name: count, dtype: int64)

In [9]:
invalid_row = train_df[train_df["extra"].notna()]
invalid_row

,image_name,text_tr,text_en,education,extra
6234,108-3B-457-M-H,Mutluluk,sevinçli bir şey sevgi,Primary,something joyful love


In [10]:
train_df.loc[train_df["extra"].notna(), "text_en"] = train_df.loc[
    train_df["extra"].notna(), "extra"
]

In [11]:
train_df[train_df["extra"].notna()][["text_en", "extra"]]

,text_en,extra
6234,something joyful love,something joyful love


In [12]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9228 entries, 0 to 9227
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   image_name  9228 non-null   object
 1   text_tr     9226 non-null   object
 2   text_en     9226 non-null   object
 3   education   9228 non-null   object
 4   extra       1 non-null      object
dtypes: object(5)
memory usage: 360.6+ KB


In [13]:
test_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1632 entries, 0 to 1631
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   image_name  1632 non-null   object 
 1   text_tr     1632 non-null   object 
 2   text_en     1632 non-null   object 
 3   education   1632 non-null   object 
 4   extra       0 non-null      float64
dtypes: float64(1), object(4)
memory usage: 63.9+ KB


In [14]:
train_df.describe(include='all')

,image_name,text_tr,text_en,education,extra
count,9228,9226,9226,9228,1
unique,9228,8892,8730,2,1
top,205-5G-1242-F-H,Mutluluk benim için doğadır,Happiness is nature for me,Secondary,something joyful love
freq,1,13,15,4614,1


In [15]:
test_df.describe(include='all')

,image_name,text_tr,text_en,education,extra
count,1632,1632,1632,1632,0.0
unique,1632,1610,1594,2,NaN
top,205-5D-417-F-H,Mutluluk benim için ailedir,I am very sad today,Secondary,NaN
freq,1,4,5,816,NaN
mean,NaN,NaN,NaN,NaN,NaN
std,NaN,NaN,NaN,NaN,NaN
min,NaN,NaN,NaN,NaN,NaN
25%,NaN,NaN,NaN,NaN,NaN
50%,NaN,NaN,NaN,NaN,NaN
75%,NaN,NaN,NaN,NaN,NaN


In [16]:
train_df.columns.values

array(['image_name', 'text_tr', 'text_en', 'education', 'extra'],
      dtype=object)

In [17]:
test_df.columns.values

array(['image_name', 'text_tr', 'text_en', 'education', 'extra'],
      dtype=object)

In [18]:
train_df = train_df.drop(columns=["text_tr", "extra"])
test_df = test_df.drop(columns=["text_tr", "extra"])

In [19]:
train_df.head()

,image_name,text_en,education
0,205-5G-1242-F-H,Being the first is a very proud and happy thin...,Secondary
1,210-5C-818-F-H,Happiness is the light in life,Secondary
2,214-6B-421-F-S,Unhappiness is never giving up after feeling s...,Secondary
3,206-6D-790-F-S,"For me, sadness is being wronged, being exclud...",Secondary
4,203-5H-720-F-H,"For me, happiness is a world where there is no...",Secondary


### Handle Missing Values

In [20]:
train_df['education'].value_counts()

education
Secondary    4614
Primary      4614
Name: count, dtype: int64

In [21]:
train_df.isnull().sum()

image_name    0
text_en       2
education     0
dtype: int64

In [22]:
missing_text = train_df[train_df["text_en"].isna()]
missing_text

,image_name,text_en,education
54,214-5K-1917-M-H,NaN,Secondary
3326,214-5K-348-M-H,NaN,Secondary


In [23]:
train_df = train_df.dropna(subset=["text_en"])
test_df  = test_df.dropna(subset=["text_en"])

## Encoding the Data

In [24]:
label_map = {"Primary": 0, "Secondary": 1}

In [25]:
train_df["label"] = train_df["education"].map(label_map)
test_df["label"]  = test_df["education"].map(label_map)

In [26]:
print(train_df["label"].value_counts())
print(test_df["label"].value_counts())

label
0    4614
1    4612
Name: count, dtype: int64
label
1    816
0    816
Name: count, dtype: int64


## Load the tokenizer

BERT - Bidirectional Encoder Representations from Transformers
bert-base-uncased - Not case sensitive 

In [27]:
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
MAX_LEN = 64

## Split the data

In [28]:
train_split, val_split = train_test_split(
    train_df,
    test_size=0.2,
    random_state=42,
    stratify=train_df["label"],
)

In [29]:
print(train_split.shape)
print(val_split.shape)

(7380, 4)
(1846, 4)


## Tokenization

In [30]:
def tokenize_texts(texts, tokenizer, max_len):
    return tokenizer(
        texts.tolist(),
        padding=True, 
        truncation=True,
        max_length=max_len,
        return_tensors="pt",
    )

In [31]:
train_encodings = tokenize_texts(train_split["text_en"], tokenizer, MAX_LEN)
val_encodings = tokenize_texts(val_split["text_en"], tokenizer, MAX_LEN)
test_encodings = tokenize_texts(test_df["text_en"], tokenizer, MAX_LEN)

In [32]:
train_encodings.keys()

KeysView({'input_ids': tensor([[  101,  1996,  3606,  ...,     0,     0,     0],
        [  101,  2108, 12421,  ...,     0,     0,     0],
        [  101,  2054,  6314,  ...,     0,     0,     0],
        ...,
        [  101,  1045,  2572,  ...,     0,     0,     0],
        [  101,  1045,  2572,  ...,     0,     0,     0],
        [  101,  8404,  2003,  ...,     0,     0,     0]]), 'token_type_ids': tensor([[0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        ...,
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        ...,
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0]])})

In [33]:
train_encodings["input_ids"].shape

torch.Size([7380, 64])

## Wrap into a PyTorch Dataset

In [34]:
class EncodedTextDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels.tolist()
        
    def __len__(self):
        return len(self.labels)
    
    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        item['label'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

### Create datasets

In [35]:
train_ds = EncodedTextDataset(train_encodings, train_split["label"])
val_ds = EncodedTextDataset(val_encodings, val_split["label"])
test_ds = EncodedTextDataset(test_encodings, test_df["label"])

In [36]:
train_ds[0].keys()

dict_keys(['input_ids', 'token_type_ids', 'attention_mask', 'label'])

In [37]:
train_ds[0]["input_ids"].shape

torch.Size([64])

## Define the BERT model

In [38]:
model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


## Training Arguments 

In [39]:
training_args = TrainingArguments(
    output_dir="./text_model",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_steps=100
)

## Add Metrics

In [41]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds)
    }

## Build Trainer

In [44]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics
)

## Training the Model 